# SYDE 556/750 &mdash; Practice Notebook 4
## Decision Making, Learning, and the LMU

**Prepares you for: Test 4.**

This notebook uses Nengo. It covers a recurrent decision-making model, the Legendre Memory Unit, online learning with the PES rule (including an adaptive controller), and using the SVD to analyse what a population can represent.

**How to use it**
- **No solutions are released.** Each part ends with an *Expected result* checkpoint so you can tell whether you are on track.
- Cells marked &#x270D; are for written answers.

**Contents**
1. An accumulate-to-threshold decision model
2. The Legendre Memory Unit
3. Online learning with the PES rule
4. Adaptive control of a pendulum
5. Analysing representation with the SVD


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import nengo


# 1. An accumulate-to-threshold decision model

A standard account of simple decision-making is that the brain gradually **accumulates evidence** until it crosses a threshold; weaker evidence takes longer. We'll build a neural version. A single input value gives the evidence; the model outputs +1 or &minus;1 to indicate its choice.


**a) Accumulation.** Build a recurrent accumulator: a 1-D ensemble of **50 neurons**, an input `Node` with constant value `[0.1]` connected in, and a recurrent connection from the ensemble back to itself computing the identity with `synapse=0.1` (so evidence accumulates fairly stably; leave the input connection's synapse at its default).

Plot the accumulator value for four inputs &mdash; 0.2, 0.1, &minus;0.1, &minus;0.2 &mdash; running each for 2 s, using a probe `synapse=0.01`.


In [ ]:
# Recurrent accumulator (50 neurons); recurrent identity with synapse=0.1.
# Plot the stored value for inputs [0.2, 0.1, -0.1, -0.2], 2 s each.
# TODO


*Expected:* the value ramps up (or down) and tends to saturate near +1 (or &minus;1); stronger inputs ramp faster.

**b) Discussion.** What is the mathematical computation here (relationship between input and output)? Why does the value stop changing once it reaches +1 or &minus;1?


&#x270D; *Your answer here.*


**c) Adding noise.** Add a random-walk component by injecting noise into the ensemble:
```python
acc.noise = nengo.processes.WhiteSignal(period=10, high=100, rms=1)
```
(replace `acc` with your ensemble's name). Reproduce the part-(a) plot with `rms` = 1, 3, 5, and 10. What happens to the output?


In [ ]:
# Add ensemble noise; reproduce the plot for rms = 1, 3, 5, 10.
# TODO


&#x270D; *Your answer here.*


**d) Adding a decision.** Add a one-dimensional output ensemble (50 neurons). Connect the accumulator to it computing
```python
def choice(x):
    if x[0] > 0.9:
        return 1
    elif x[0] < -0.9:
        return -1
    else:
        return 0
```
so the output stays near 0 until the accumulator is large enough, then snaps to +1 or &minus;1. Use noise `rms=3` and probe `synapse=0.01`. Plot the accumulator and the decision output for the four input values. Does it decide faster with stronger evidence? What differs between the requested computation and the actual result?


In [ ]:
def choice(x):
    if x[0] > 0.9:
        return 1
    elif x[0] < -0.9:
        return -1
    else:
        return 0

# Add the output ensemble + choice connection; plot accumulator and decision for the four inputs.
# TODO


&#x270D; *Your answer here.*


**e) Combining ensembles.** Replace the two 1-D ensembles with a single **2-D** ensemble (use `ens[0]`/`ens[1]` to target dimensions), with 100 neurons and then 500 neurons; raise the `radius` appropriately. How does performance change? What biological claim changes when the two values share one ensemble, and how might you tell experimentally whether the brain keeps them separate or combined?


&#x270D; *Your answer (plus plots).*


In [ ]:
# 2-D combined version with 100 and 500 neurons.
# TODO


**f) Improving representation.** Back to the two-ensemble version: set the output ensemble's `intercepts = nengo.dists.Uniform(0.4, 0.9)`. Reproduce the part-(d) plot. How does this change performance, and why does the output now sit at exactly zero until the decision is made? Why couldn't you use this trick in the combined (2-D) version?


In [ ]:
# Reproduce the part-(d) plot with `intercepts = nengo.dists.Uniform(0.4, 0.9)`.
# TODO


&#x270D; *Your answer (plus plots).*


# 2. The Legendre Memory Unit

The LMU stores a window of recent input so we can compute functions of the input *over time* rather than just its current value. We'll use it to tell a 1 Hz sine wave from a 2 Hz sine wave, which is impossible from a single instantaneous sample.


**a) Representing information over time.** The LMU computes $\frac{dx}{dt} = Ax + Bu$ with the $A$, $B$ below. Implement it in Nengo as a single $q$-dimensional ensemble (**1000 neurons**), with `synapse=0.1` on both the recurrent and input connections. Use `theta=0.5`, `q=6`.

Drive it with a 1 Hz sine for the first 2 s and a 2 Hz sine for the next 2 s, run 4 s, and plot the 6-D $x$ with a probe `synapse=0.01`. There should be a clear change at $t = 2$ s.


In [ ]:
# LMU A, B matrices (given):
q = 6
theta = 0.5
A = np.zeros((q, q))
B = np.zeros((q, 1))
for i in range(q):
    B[i] = (-1.)**i * (2*i + 1)
    for j in range(q):
        A[i, j] = (2*i + 1) * (-1 if i < j else (-1.)**(i - j + 1))
A = A / theta
B = B / theta

stim = nengo.Node(lambda t: np.sin(2*np.pi*t) if t < 2 else np.sin(2*np.pi*t*2))

# Build a q-dimensional ensemble (1000 neurons) implementing dx/dt = A x + B u
# via a recurrent connection (remember to convert A, B to the A', B' for synapse=0.1).
# Probe x (synapse=0.01); run 4 s; plot the 6-D state.
# TODO


**b) Computing the function.** Add a 1-D, 50-neuron ensemble for the category. You already have the $x$ values from part (a); the target is +1 for $t \in [0, 2)$ and &minus;1 for $t \in [2, 4)$. Build the target vector (shape `(4000, 1)`: first 2000 = 1, last 2000 = &minus;1) and make the connection with those as evaluation points:
```python
nengo.Connection(a, b, eval_points=x_values, function=target)
```
Plot the resulting category (probe `synapse=0.01`). It should be near +1 for the first 2 s and near &minus;1 for the second 2 s (not perfect!).


In [ ]:
# Build target; connect the LMU ensemble to a 1-D category ensemble via eval_points/function; plot.
# TODO


**c) Swapping the input order.** Repeat (b) with a 2 Hz sine first and a 1 Hz sine second. How well does it do? You may notice it misclassifies for roughly the first 0.25 s. Why does that happen, and what could you change to fix it?


&#x270D; *Your answer here.*


**d) Number of neurons.** Repeat (b) varying the neuron count in the LMU ensemble: 50, 100, 200, 500, 1000, 2000, 5000. Show how the behaviour changes and plot the classification RMSE vs. neuron count. Explain why you think you observe that change in RMSE.

In [ ]:
# Sweep neuron count; plot classification RMSE.
# TODO


&#x270D; *Your answer here.*


**e) The $q$ value.** Repeat (b) (back to 1000 neurons) varying $q$: 1, 2, 4, 8, 16, 32, 64. Show how the behaviour changes and plot the classification RMSE vs. $q$. Explain why you think you observe that change in RMSE.

In [ ]:
# Sweep q; plot classification RMSE.
# TODO


&#x270D; *Your answer here.*


# 3. Online learning with the PES rule

Usually we compute connection weights once and leave them fixed. Instead we can adjust them *online* with a learning rule, which changes the function being computed. The **PES** rule takes an extra **error signal** indicating whether the output should increase or decrease.


**a) Basic online learning.** Build a network that learns the identity function. You need three 1-D ensembles (input, output, and error) each with **200 neurons**. Drive the input with a 2 Hz band-limited white-noise signal:
```python
stim = nengo.Node(nengo.processes.WhiteSignal(period=100, high=2, rms=0.3))
```
Initialize the learning connection to compute zero and use the PES rule:
```python
def initialization(x):
    return 0
c = nengo.Connection(pre, post, function=initialization,
                     learning_rule_type=nengo.PES(learning_rate=1e-4))
```
The error ensemble should compute (output &minus; desired); for the identity, the desired output is the input. Then drive the learning rule:
```python
nengo.Connection(error, c.learning_rule)
```
(Leave the connection synapses at their defaults.) Run for 10 s and plot the input and the resulting output (probe `synapse=0.01`).


In [ ]:
# Three ensembles (input, output, error); PES learning connection; learn the identity.
# Run 10 s; plot input vs output.
# TODO


*Expected:* the output is poor at first and tracks the input fairly well after the first few seconds.

**b) Error sign.** What would happen if you reversed the sign of the error calculation (computing `target - output` rather than `output - target`)? Why?


&#x270D; *Your answer here.*


**c) Computing metrics.** Break the data into 2-second chunks and compute the RMSE between the target (the stimulus) and the output for each chunk; for a 10 s run that gives 5 values. Repeat the simulation 10 times and plot the average of each. The model should improve over time but not reach zero error.


In [ ]:
# 5 RMSE values (2-second chunks); average over 10 runs; plot.
# TODO


*Expected:* RMSE falls across the five chunks and levels off above zero.

**d) Increasing learning time.** Repeat (c) but run for 100 s instead of 10 s. How do the results change?

In [ ]:
# Repeat (c) for a 100 s run.
# TODO


**e) Learning rate.** Repeat (d) but decrease the learning rate to `1e-5`. How do the results change, and how do they compare to (c)?


&#x270D; *Your answer (plus plots).*


**f) Improving performance.** If you wanted the learned result to be even more accurate, what would you change about the model and the learning process?


&#x270D; *Your answer here.*


**g) Learning other functions.** Repeat (a) but learn a function where the input is a scalar $x$ and the output is the vector $[x^2, -x]$. This means changing the dimensionality of some ensembles and adding a `function=` on the connection from the stimulus to the error.


In [ ]:
# Learn the vector function [x^2, -x] via PES.
# TODO


# 4. Adaptive control of a pendulum

PES learning becomes useful in adaptive control. A simple PD controller can drive a pendulum toward a target angle, but it doesn't know about gravity (an unmodeled force), so it settles with a steady-state offset. We'll add an adaptive ensemble that learns online, via the PES rule, to supply the extra torque that cancels the unmodeled force.

The plant is a 1-link pendulum. We measure the angle $\theta$ from the horizontal, with $\theta = 0$ pointing to the right (positive $\theta$ = counter-clockwise, so the arm rotates up). With this convention, gravity produces its *maximum* torque at $\theta = 0$, so keeping the pendulum horizontal genuinely requires a bias torque that the adaptive network can learn. 

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 380 240" width="380" height="240" font-family="sans-serif" font-size="13">
  <!-- Pivot -->
  <circle cx="150" cy="130" r="4" fill="#333"/>
  <text x="146" y="149" text-anchor="end" fill="#555">pivot</text>
  <!-- Horizontal reference (theta = 0) pointing right -->
  <line x1="150" y1="130" x2="300" y2="130" stroke="#4a90d9" stroke-width="1.2" stroke-dasharray="5 3"/>
  <text x="305" y="134" fill="#4a90d9">&#952; = 0 (target: horizontal right)</text>
  <!-- Pendulum arm at theta = 45 deg above horizontal -->
  <line x1="150" y1="130" x2="221" y2="59" stroke="#222" stroke-width="2.5"/>
  <circle cx="221" cy="59" r="10" fill="#c94e4e" stroke="#7a2828" stroke-width="1.5"/>
  <text x="230" y="52" fill="#333">mass m</text>
  <!-- Angle arc from horizontal to arm -->
  <path d="M 200 130 A 50 50 0 0 0 185 95" fill="none" stroke="#333" stroke-width="1"/>
  <text x="205" y="118" fill="#333">&#952;</text>
  <!-- Gravity vector -->
  <line x1="70" y1="60" x2="70" y2="130" stroke="#666" stroke-width="1.6"/>
  <path d="M 66 126 L 70 134 L 74 126 z" fill="#666"/>
  <text x="60" y="55" fill="#666">g</text>
  <!-- Arm length label -->
  <text x="175" y="105" fill="#555" font-style="italic">L</text>
</svg>

The state is $[\theta, \dot\theta]$ and the dynamics are
$$ m L^2\,\ddot\theta = \tau - m g L \cos\theta, $$
where $\tau$ is the applied torque. The $\cos\theta$ term is the projection of gravity perpendicular to the arm; it is maximal at $\theta = 0$ (arm horizontal) and vanishes at $\theta = \pm\pi/2$ (arm vertical). (This `plant` Node is given; the control is up to you.)

&#x1F4D6; This mirrors the `pendulum_pd_adaptive.py` example referenced in the course README.


In [ ]:
# Given plant: a 1-link pendulum integrated with simple Euler steps.
# theta is measured from the horizontal (theta = 0 = arm pointing right),
# so gravity produces a torque -m*g*L*cos(theta), maximal at theta = 0.
g, L, m = 9.8, 1.0, 1.0

def make_pendulum(dt=0.001, theta0=np.pi/4):
    state = np.array([theta0, 0.0])   # [angle, angular velocity]
    def plant(t, u):
        torque = u[0]
        theta, omega = state
        omega_dot = (torque - m*g*L*np.cos(theta)) / (m*L**2)
        state[0] = theta + dt*omega
        state[1] = omega + dt*omega_dot
        return state.copy()
    return plant


**a) PD control only.** Drive the pendulum to a target angle (say $\theta^* = 0$) with a PD controller, $\tau_{PD} = k_p(\theta^* - \theta) - k_d\,\dot\theta$, and **no** adaptation. Plot the angle over time. You should see a persistent steady-state error because gravity is unmodeled.


In [ ]:
# Plant Node + PD controller (choose kp, kd); no learning. Plot angle vs target.
# TODO


*Expected:* the angle approaches but does not reach the target. There is a steady offset set by gravity.

**b) Add an adaptive ensemble.** Add an ensemble (a few hundred to ~1000 neurons) that represents the pendulum state and connects to the torque through a PES-learning connection initialized to zero. Use the PD signal as the error driving the learning rule, so the adaptive ensemble gradually takes over supplying the bias torque. Plot the angle over time and show the steady-state error shrinks as learning proceeds.


In [ ]:
# Adaptive ensemble representing the state; PES-learned connection to torque;
# error = PD signal. Compare angle-vs-time to part (a).
# TODO


*Expected:* over time the adaptive signal cancels the gravity term and the steady-state error collapses toward zero.

**c) Discussion.** What is the adaptive ensemble learning to represent? How is this the same mechanism as the identity-learning in &sect;3, and what plays the role of the "error" here?


&#x270D; *Your answer here.*


# 5. Analysing representation with the SVD

The singular value decomposition tells us which functions a population can represent well. Here you'll compute it and connect the math to the representation.


**a) Singular value spectrum.** Build a 1-D population (e.g., 100 LIF neurons over $x \in [-1, 1]$), form its activity matrix $A$ (neurons &times; samples), and compute the SVD. Plot the singular value spectrum (log scale).


In [ ]:
# Build A over sampled x; compute SVD; plot the singular values (log scale).
# TODO


**b) Which matrix, and why.** In class we used the SVD to understand what functions a population can compute. What matrix did we decompose, and why is that the right matrix? Name one way of understanding what the decomposition tells you about it.


&#x270D; *Your answer here.*


**c) Noise and the SVD.** Overlay an estimate of the noise floor (set by the representation noise $\sigma$) on your spectrum. What does the relationship between the noise variance and the size of a given singular value tell you about whether that dimension, and the functions relying on it, is usable? Use equations.


&#x270D; *Your answer here.*


*Expected:* singular values that fall below the noise floor correspond to dimensions the population cannot reliably represent; those functions can't be decoded above the noise.


---
*End of Practice Notebook 4.* Test 4 will assume you can do everything above by hand or from scratch.
